# Detecting hidden commercial activity in consumer card transactions
**Mastercard Data Quest 2026**

Some individual cardholders run actual businesses through personal cards. They pay for ads, cloud subscriptions, wholesale goods — but the bank sees them as regular consumers. The goal here is to identify that group from transaction behavior alone.

The setup: business cardholders serve as labeled positives (label = 1). Their spending patterns are distinct enough to train a classifier, which then scores all 80K consumer cards.

Data is fully synthetic, covering October 2025 through March 2026.

In [ ]:
# install if running in Colab
# !pip install lightgbm shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import entropy as scipy_entropy
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
import lightgbm as lgb
import shap
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
SEED = 42
np.random.seed(SEED)

## 1. Data loading

Three files:
- `business_cards_MDQ.parquet` — ~3M transactions from 25K business cards
- `consumer_cards_MDQ.parquet` — ~9.8M transactions from 80K consumer cards
- `merchants_reference.parquet` — 2165 merchants with MCC codes and metadata

All files live in Google Drive. Mount once, then read.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/MDQ_26 Case Information/"

biz = pd.read_parquet(BASE + "business_cards_MDQ.parquet")
con = pd.read_parquet(BASE + "consumer_cards_MDQ.parquet")
mer = pd.read_parquet(BASE + "merchants_reference.parquet")

print(f"business : {len(biz):>10,} txns  |  {biz['card_number'].nunique():,} cards")
print(f"consumer : {len(con):>10,} txns  |  {con['card_number'].nunique():,} cards")
print(f"merchants: {len(mer):>10,} rows")

## 2. Data quality

`MER_000000` shows up with two different MCCs in transaction data: 7311 (Advertising) and 7012 (Hotels). The reference table lists only 7311, and MCC 7012 makes no sense for a merchant labeled Google Ads.

Top-merchant counts confirm the suspicion. MER_000000 dominates both segments at roughly the same per-card rate, which is impossible if it's purely Google Ads. Consumers don't buy ads at consumer-card volumes. The 7012 rows behave like a placeholder bucket for transactions that lost their real merchant ID somewhere in the pipeline.

Drop those rows entirely. Genuine Google Ads transactions (mcc=7311) stay, the noise goes. After the drop, top-merchant lists become exactly what you'd expect: ads + B2B SaaS for business cards, telecom + gas + food for consumers.

In [ ]:
# check nulls
assert biz.isnull().sum().sum() == 0, "nulls in business"
assert con.isnull().sum().sum() == 0, "nulls in consumer"

# cards must not overlap between segments
overlap = set(biz["card_number"]) & set(con["card_number"])
assert len(overlap) == 0, f"card overlap found: {len(overlap)}"

# all merchant_ids must be in the reference
assert set(biz["merchant_id"]).issubset(set(mer["merchant_id"]))
assert set(con["merchant_id"]).issubset(set(mer["merchant_id"]))

print("sanity checks passed")

# drop MER_000000 + mcc=7012 placeholder rows (see markdown above)
biz_before, con_before = len(biz), len(con)
placeholder = lambda df: (df["merchant_id"] == "MER_000000") & (df["mcc"] == "7012")
biz = biz.loc[~placeholder(biz)].reset_index(drop=True)
con = con.loc[~placeholder(con)].reset_index(drop=True)

print(f"dropped placeholder rows: business {biz_before - len(biz):,}, "
      f"consumer {con_before - len(con):,}")
print(f"remaining: business {len(biz):,} txns, consumer {len(con):,} txns")

In [ ]:
# parse timestamps once, reuse everywhere
for df in [biz, con]:
    df["ts"]               = pd.to_datetime(df["transaction_timestamp"])
    df["hour"]             = df["ts"].dt.hour
    df["dow"]              = df["ts"].dt.dayofweek   # 0 = Monday
    df["month"]            = df["ts"].dt.to_period("M")
    df["is_weekend"]       = df["dow"] >= 5
    df["is_biz_hours"]     = df["hour"].between(9, 18)  # 9am-6pm inclusive

## 3. EDA

The two segments differ across every behavioral dimension. What follows documents the main signals that will drive the model.

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Cards", "Transactions",
        "Median txn amount (KZT)", "Mean txn amount (KZT)",
        "Online ratio", "Recurring ratio",
        "Business hours ratio (9-18)", "Weekend ratio",
        "Tokenized ratio",
    ],
    "Business": [
        f"{biz['card_number'].nunique():,}",
        f"{len(biz):,}",
        f"{biz['transaction_amount_kzt'].median():,.0f}",
        f"{biz['transaction_amount_kzt'].mean():,.0f}",
        f"{(biz['channel']=='online').mean():.1%}",
        f"{biz['is_recurring'].mean():.1%}",
        f"{biz['is_biz_hours'].mean():.1%}",
        f"{biz['is_weekend'].mean():.1%}",
        f"{biz['tokenized'].mean():.1%}",
    ],
    "Consumer": [
        f"{con['card_number'].nunique():,}",
        f"{len(con):,}",
        f"{con['transaction_amount_kzt'].median():,.0f}",
        f"{con['transaction_amount_kzt'].mean():,.0f}",
        f"{(con['channel']=='online').mean():.1%}",
        f"{con['is_recurring'].mean():.1%}",
        f"{con['is_biz_hours'].mean():.1%}",
        f"{con['is_weekend'].mean():.1%}",
        f"{con['tokenized'].mean():.1%}",
    ],
})
display(summary)

In [ ]:
# transaction amount distributions — log scale to see both segments on one plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, label, color in zip(
    axes,
    [biz, con],
    ["Business cards", "Consumer cards"],
    ["#c0392b", "#2980b9"]
):
    vals = np.log1p(df["transaction_amount_kzt"])
    ax.hist(vals, bins=60, color=color, alpha=0.82, edgecolor="white", linewidth=0.3)
    med = df["transaction_amount_kzt"].median()
    ax.axvline(np.log1p(med), color="black", linestyle="--", linewidth=1.3,
               label=f"median = {med:,.0f} KZT")
    ax.set_title(label)
    ax.set_xlabel("log(amount + 1)")
    ax.set_ylabel("transaction count")
    ax.legend()

plt.suptitle("Transaction amount distributions", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# hourly activity — this is one of the cleaner separating signals
biz_hour = biz.groupby("hour").size() / len(biz) * 100
con_hour = con.groupby("hour").size() / len(con) * 100

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(biz_hour.index, biz_hour.values, marker="o", label="Business", color="#c0392b", linewidth=2)
ax.plot(con_hour.index, con_hour.values, marker="o", label="Consumer", color="#2980b9", linewidth=2)
ax.axvspan(9, 18, alpha=0.07, color="green", label="Working hours (9-18)")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Share of all transactions (%)")
ax.set_title("Transaction activity by hour")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

print(f"Business txns in 9-18h : {biz['is_biz_hours'].mean():.1%}")
print(f"Consumer txns in 9-18h : {con['is_biz_hours'].mean():.1%}")

In [ ]:
# monthly distribution — consumer has a December spike (~20% vs ~16% other months)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, label, color in zip(
    axes, [biz, con], ["Business", "Consumer"], ["#c0392b", "#2980b9"]
):
    monthly = df.groupby("month").size()
    pct = monthly / monthly.sum() * 100
    ax.bar([str(m) for m in pct.index], pct.values, color=color, alpha=0.82, edgecolor="white")
    ax.axhline(100/6, color="black", linestyle="--", linewidth=1.1, label="uniform baseline")
    ax.set_title(f"{label} — monthly share (%)")
    ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=30)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# B2B MCC prevalence — how much more common each code is in business vs consumer
b2b_check = {
    "7399  Business Services NEC" : "7399",
    "5045  Computers & Peripherals": "5045",
    "5943  Office Supplies"        : "5943",
    "8931  Accounting Services"    : "8931",
    "7372  Software / Cloud"       : "7372",
    "4816  Internet Services"      : "4816",
    "7311  Advertising"            : "7311",
}

rows = []
for label, mcc in b2b_check.items():
    bp = (biz["mcc"] == mcc).mean() * 100
    cp = (con["mcc"] == mcc).mean() * 100
    rows.append({"MCC": label, "Business %": bp, "Consumer %": cp,
                 "Ratio (biz/con)": bp / cp if cp > 0 else np.inf})

mcc_df = pd.DataFrame(rows).sort_values("Ratio (biz/con)", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(mcc_df["MCC"], mcc_df["Ratio (biz/con)"], color="#c0392b", alpha=0.82)
for bar, val in zip(bars, mcc_df["Ratio (biz/con)"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.0f}x", va="center", fontsize=10)
ax.set_xlabel("Business / Consumer ratio")
ax.set_title("How much more common each B2B MCC is in business cards")
plt.tight_layout()
plt.show()

display(mcc_df)

## 4. Feature engineering

Everything is aggregated at the card level — one row per `card_number`. The model never sees individual transactions.

Feature groups:
- **Amount** — volume, average, variability, large-ticket share
- **Channel / payment type** — online ratio, recurring, tokenized
- **Time** — business hours concentration, weekend activity, monthly stability
- **Merchant diversity** — unique merchants, MCC entropy, geographic spread
- **B2B MCC ratios** — share of spend in advertising, cloud/software, wholesale, professional services, etc.

The B2B MCC groups come directly from the segment analysis above. Each ratio captures a specific type of commercial spend that consumers rarely do.

In [ ]:
# join recurring_capable from merchant reference — tells us if a card
# regularly pays subscription-style merchants (cloud, SaaS, etc.)
mer_slim = mer[["merchant_id", "recurring_capable"]].copy()
biz = biz.merge(mer_slim, on="merchant_id", how="left")
con = con.merge(mer_slim, on="merchant_id", how="left")

assert biz["recurring_capable"].isna().sum() == 0
assert con["recurring_capable"].isna().sum() == 0
print("merchant join OK")

In [ ]:
B2B_MCC_GROUPS = {
    "advertising"          : ["7311", "7319"],
    "cloud_software"       : ["7372", "5968", "4816"],
    "business_services"    : ["7399", "7392"],
    "office_supplies"      : ["5111", "5943"],
    "wholesale"            : ["5045", "5046", "5065", "5085", "5099"],
    "professional_services": ["8931", "8111", "8911", "7361"],
    "logistics"            : ["4214", "4215", "4111", "4121"],
}


def build_card_features(df):
    g = df.groupby("card_number")

    # base amount aggregates
    feats = g["transaction_amount_kzt"].agg(
        total_amount="sum",
        avg_amount="mean",
        median_amount="median",
        max_amount="max",
        std_amount="std",
        txn_count="count",
    ).reset_index()

    feats["cv_amount"] = feats["std_amount"] / feats["avg_amount"]

    # share of transactions above 100K KZT — business-sized purchases
    feats["large_txn_ratio"] = (
        g["transaction_amount_kzt"].apply(lambda x: (x > 100_000).mean()).values
    )
    # round amounts sometimes signal invoicing behavior
    feats["round_amount_ratio"] = (
        g["transaction_amount_kzt"].apply(lambda x: (x % 1000 == 0).mean()).values
    )

    # channel and payment method
    feats["online_ratio"]            = g["channel"].apply(lambda x: (x == "online").mean()).values
    feats["recurring_ratio"]         = g["is_recurring"].mean().values
    feats["tokenized_ratio"]         = g["tokenized"].mean().values
    feats["recurring_capable_ratio"] = g["recurring_capable"].mean().values

    # time features
    feats["biz_hours_ratio"] = g["is_biz_hours"].mean().values
    feats["weekend_ratio"]   = g["is_weekend"].mean().values
    feats["active_days"]     = g["ts"].apply(lambda x: x.dt.date.nunique()).values
    feats["txn_per_day"]     = feats["txn_count"] / feats["active_days"]
    feats["months_active"]   = g["month"].nunique().values

    # how stable is monthly spending across the 6-month window
    def monthly_cv(grp):
        ms = grp.resample("ME", on="ts")["transaction_amount_kzt"].sum()
        return ms.std() / ms.mean() if ms.mean() > 0 else 0

    feats["monthly_amount_cv"] = g.apply(monthly_cv, include_groups=False).values

    # merchant and MCC diversity
    feats["unique_merchants"] = g["merchant_id"].nunique().values
    feats["unique_mcc"]       = g["mcc"].nunique().values
    feats["unique_countries"] = g["country"].nunique().values

    feats["mcc_entropy"] = g["mcc"].apply(
        lambda x: scipy_entropy(x.value_counts(normalize=True))
    ).values
    feats["merchant_entropy"] = g["merchant_id"].apply(
        lambda x: scipy_entropy(x.value_counts(normalize=True))
    ).values

    # B2B MCC group ratios
    for group_name, codes in B2B_MCC_GROUPS.items():
        feats[f"{group_name}_ratio"] = g["mcc"].apply(
            lambda x: x.isin(codes).mean()
        ).values

    # composite: sum of all B2B group ratios — single most discriminating feature
    b2b_cols = [f"{k}_ratio" for k in B2B_MCC_GROUPS]
    feats["b2b_composite_ratio"] = feats[b2b_cols].sum(axis=1)

    return feats


print("building business features...")
biz_feats = build_card_features(biz)
biz_feats["label"] = 1

print("building consumer features...")
con_feats = build_card_features(con)
con_feats["label"] = 0

print(f"business feature matrix : {biz_feats.shape}")
print(f"consumer feature matrix : {con_feats.shape}")

In [ ]:
# quick sanity: are the features actually discriminating?
feature_cols = [c for c in biz_feats.columns if c not in ["card_number", "label"]]

comparison = pd.DataFrame({
    "Business (median)": biz_feats[feature_cols].median(),
    "Consumer (median)": con_feats[feature_cols].median(),
})
comparison["Ratio B/C"] = (
    comparison["Business (median)"] / comparison["Consumer (median)"]
).replace([np.inf, -np.inf], np.nan).round(2)

display(comparison.sort_values("Ratio B/C", ascending=False).head(20))

In [ ]:
# distribution plots for top features
top_feats = [
    "b2b_composite_ratio", "online_ratio", "biz_hours_ratio",
    "avg_amount", "recurring_ratio", "cloud_software_ratio",
    "advertising_ratio", "weekend_ratio", "recurring_capable_ratio",
]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for ax, feat in zip(axes, top_feats):
    cap = max(biz_feats[feat].quantile(0.99), con_feats[feat].quantile(0.99))
    ax.hist(con_feats[feat].clip(upper=cap), bins=40, alpha=0.6,
            color="#2980b9", label="Consumer", density=True)
    ax.hist(biz_feats[feat].clip(upper=cap), bins=40, alpha=0.6,
            color="#c0392b", label="Business", density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Feature distributions: Business vs Consumer (per-card aggregates)",
             y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 5. Model training

**Why this framing works:** business cards are a clean proxy for commercial behavior. Training the model to separate them from consumers teaches it to recognize the exact patterns we care about.

**Training set:** all 25K business cards (label=1) + 50K randomly sampled consumer cards (label=0). The remaining 30K consumers are held out purely for scoring.

**Models:**
- Logistic Regression — interpretable baseline, tells us whether the features are linearly separable
- LightGBM — main model; handles feature interactions, class weighting built in

Class imbalance: `scale_pos_weight = 50000 / 25000 = 2`. This is lighter than the true population ratio but prevents the model from being overly conservative.

In [ ]:
# build training set
con_sample = con_feats.sample(n=50_000, random_state=SEED)
train_df = pd.concat([biz_feats, con_sample], ignore_index=True)

X = train_df[feature_cols].values
y = train_df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"train: {X_train.shape}  |  pos rate: {y_train.mean():.3f}")
print(f"test : {X_test.shape}   |  pos rate: {y_test.mean():.3f}")

In [ ]:
# baseline: logistic regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
lr.fit(X_train_sc, y_train)

lr_proba = lr.predict_proba(X_test_sc)[:, 1]
print(f"Logistic Regression  |  ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}  "
      f"PR-AUC: {average_precision_score(y_test, lr_proba):.4f}")

In [ ]:
# LightGBM — main model
lgb_params = {
    "objective"       : "binary",
    "metric"          : ["auc", "binary_logloss"],
    "num_leaves"      : 63,
    "learning_rate"   : 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq"    : 5,
    "min_child_samples": 30,
    "scale_pos_weight": 2.0,   # 50K neg / 25K pos
    "lambda_l1"       : 0.1,
    "lambda_l2"       : 1.0,
    "verbose"         : -1,
    "seed"            : SEED,
}

dtrain = lgb.Dataset(X_train, label=y_train,
                     feature_name=feature_cols)
dval   = lgb.Dataset(X_test,  label=y_test,
                     feature_name=feature_cols, reference=dtrain)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100),
]

lgb_model = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dval],
    callbacks=callbacks,
)

lgb_proba = lgb_model.predict(X_test)
print(f"\nLightGBM  |  ROC-AUC: {roc_auc_score(y_test, lgb_proba):.4f}  "
      f"PR-AUC: {average_precision_score(y_test, lgb_proba):.4f}")

## 6. Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

RocCurveDisplay.from_predictions(y_test, lgb_proba, ax=axes[0],
                                  name="LightGBM", color="#c0392b")
RocCurveDisplay.from_predictions(y_test, lr_proba, ax=axes[0],
                                  name="LogReg baseline", color="#7f8c8d")
axes[0].set_title("ROC curve")

PrecisionRecallDisplay.from_predictions(y_test, lgb_proba, ax=axes[1],
                                         name="LightGBM", color="#c0392b")
PrecisionRecallDisplay.from_predictions(y_test, lr_proba, ax=axes[1],
                                         name="LogReg baseline", color="#7f8c8d")
axes[1].set_title("Precision-Recall curve")

plt.tight_layout()
plt.show()

In [ ]:
# threshold: pick the point that maximizes F1 on validation
from sklearn.metrics import f1_score

thresholds = np.linspace(0.1, 0.9, 81)
f1_scores  = [f1_score(y_test, (lgb_proba >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best threshold (max F1 on test): {best_threshold:.2f}")

y_pred = (lgb_proba >= best_threshold).astype(int)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Consumer", "Business"]))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["Pred: Consumer", "Pred: Business"],
            yticklabels=["True: Consumer", "True: Business"],
            ax=ax)
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP — what the model actually uses to make decisions
explainer   = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test[:500])  # 500 samples is enough for the plot

shap.summary_plot(
    shap_values, X_test[:500],
    feature_names=feature_cols,
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title("SHAP feature importance (mean |SHAP value|)")
plt.tight_layout()
plt.show()

shap.summary_plot(
    shap_values, X_test[:500],
    feature_names=feature_cols,
    max_display=20,
    show=False
)
plt.title("SHAP beeswarm — feature direction and magnitude")
plt.tight_layout()
plt.show()

## 7. Scoring consumer cards

The model scores all 80K consumer cards. The score is used as a ranked prioritization list: the operational cut can be set as a top-N segment, with the top 5% shown as the main outreach cohort.

In [ ]:
con_X = con_feats[feature_cols].values
con_feats["entrepreneur_score"] = lgb_model.predict(con_X)

top_share = 0.05
top_n = int(len(con_feats) * top_share)
top_score_cutoff = con_feats["entrepreneur_score"].quantile(1 - top_share)

print("Score distribution across all 80K consumer cards:")
print(con_feats["entrepreneur_score"].describe())
print(f"\nOperational selection: top {top_share:.0%} = {top_n:,} cards")
print(f"Top-5% score cutoff (95th percentile): {top_score_cutoff:.4f}")

# score distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(con_feats["entrepreneur_score"], bins=80, color="#2980b9", alpha=0.8, edgecolor="white")
ax.axvline(top_score_cutoff, color="red", linestyle="--", linewidth=1.5,
           label=f"top 5% cutoff = {top_score_cutoff:.4f}")
ax.set_xlabel("Entrepreneur probability score")
ax.set_ylabel("Number of cards")
ax.set_title("Score distribution — 80K consumer cards")
ax.legend()
plt.tight_layout()
plt.show()

# rank-based outreach sizes are more useful than absolute score thresholds
rank_rows = []
for n in [100, 500, 1_000, top_n]:
    cutoff = con_feats["entrepreneur_score"].nlargest(n).min()
    rank_rows.append({
        "Rank segment": f"Top {n:,} cards",
        "Cards": n,
        "Share of consumer base": f"{n / len(con_feats):.2%}",
        "Minimum score in segment": cutoff,
    })

rank_summary = pd.DataFrame(rank_rows)
display(rank_summary)

In [ ]:
# profile the top-scored consumers
top_n = int(len(con_feats) * 0.05)  # top 5%
top_cards = con_feats.nlargest(top_n, "entrepreneur_score")

print(f"Top 5% by score: {len(top_cards)} cards")
print("\nMedian feature values — top 5% vs all consumers vs business cards:\n")

profile_cols = [
    "entrepreneur_score", "total_amount", "avg_amount",
    "online_ratio", "biz_hours_ratio", "weekend_ratio",
    "b2b_composite_ratio", "advertising_ratio", "cloud_software_ratio",
    "recurring_ratio", "unique_mcc",
]

biz_feats_score = biz_feats.copy()
biz_feats_score["entrepreneur_score"] = np.nan

profile = pd.DataFrame({
    "Top 5% consumers"  : top_cards[profile_cols].median(),
    "All consumers"     : con_feats[profile_cols].median(),
    "Business cards"    : biz_feats_score[profile_cols].median(),
})
display(profile.round(4))

## 8. Business value

The table below estimates the revenue opportunity if identified hidden entrepreneurs are converted to business card products.

Assumptions:
- Business card interchange is ~1.8% vs ~1.2% for consumer (standard KZ market spread)
- Conversion rate to business product: 20% (conservative)
- Median turnover for top-5% scored consumers: taken from the feature table above

These numbers are directional — the point is to show the bank where to focus outreach.

In [ ]:
median_turnover   = top_cards["total_amount"].median()   # KZT per card per 6 months
annual_turnover   = median_turnover * 2                   # annualize
n_identified      = len(top_cards)
conversion_rate   = 0.20
interchange_uplift = 0.018 - 0.012                        # 60 bps

n_converted        = int(n_identified * conversion_rate)
revenue_uplift_kzt = n_converted * annual_turnover * interchange_uplift
revenue_uplift_usd = revenue_uplift_kzt / 500             # approximate KZT/USD

print(f"Identified hidden entrepreneurs (top 5%): {n_identified:,} cards")
print(f"Estimated conversions at {conversion_rate:.0%}           : {n_converted:,} cards")
print(f"Median annual turnover per card            : {annual_turnover:,.0f} KZT")
print(f"Interchange uplift (60 bps)                : {interchange_uplift:.1%}")
print(f"\nEstimated annual revenue uplift:")
print(f"  {revenue_uplift_kzt:>15,.0f} KZT")
print(f"  {revenue_uplift_usd:>15,.0f} USD")

In [ ]:
# save scored consumer table
out = con_feats[["card_number", "entrepreneur_score"] + feature_cols].sort_values(
    "entrepreneur_score", ascending=False
)
out.to_csv(BASE + "consumer_scored.csv", index=False)
print(f"Saved {len(out):,} rows to consumer_scored.csv")

## Limitations

- **Proxy labels:** business cardholders are not identical to hidden entrepreneurs. Some business cards belong to large corporates; some hidden entrepreneurs may spend conservatively. The model approximates the target, it does not define it.
- **No incoming transactions:** the data only covers outgoing card spend. Incoming payments — a strong signal for commercial activity — are not available here.
- **Synthetic data:** patterns are clean and well-separated by design. Real-world performance will likely be lower. Cross-validation on real labeled data (e.g., customers who later upgraded to business cards) would give a more reliable estimate.
- **Static window:** features cover a fixed 6-month period. Seasonal businesses or recently started operations may score lower than they should.